# Complexity-03c — Naviguer le Zoo exécutable

[Source Python](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) · [Index](README.md) · [03b](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb)

**Objectifs** : comparer recherche et certificat, exécuter 2-SAT et 3-SAT, et expérimenter des oracles finis. **Prérequis** : graphes et formules propositionnelles. **Durée pédagogique** : 45 minutes.

Le notebook Python versionné est la source consultée. Les classes et théorèmes historiques y sont rapportés ; les expériences finies ci-dessous illustrent des mécanismes, elles ne prouvent pas des inclusions de classes ni des séparations d'oracles. La graine .NET ne reproduit pas les tirages Python.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;
using System.Globalization;
// Les sorties numériques et les coordonnées des figures gardent le point décimal entre soumissions.
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;
Console.WriteLine("Structures et générateur .NET prêts");

Structures et générateur .NET prêts


## 1. Atteignabilité et certificat

DFS conserve une pile et un ensemble de sommets visités ; un vérificateur lit un chemin fourni une seule fois. Les deux expériences utilisent des graphes aléatoires aux tailles 40, 100, 250 et 600.

In [2]:
var rng=new Random(20260925);
// Chaque sommet reçoit exactement le nombre d’arcs demandé ; les boucles et doublons restent possibles.
List<int>[] Graphe(int n,int degre) {
    var a=Enumerable.Range(0,n).Select(_=>new List<int>()).ToArray();
    for(int u=0;u<n;u++) for(int i=0;i<degre;i++) a[u].Add(rng.Next(n));
    return a;
}
// DFS compte sa pile maximale ; le résultat ne comprend pas le coût du certificat externe.
(bool,int) Dfs(List<int>[] a,int source,int cible) {
    var vus=new HashSet<int>{source}; var pile=new Stack<int>(); pile.Push(source); int max=1;
    while(pile.Count>0) { int u=pile.Pop(); if(u==cible) return (true,max);
        foreach(int v in a[u]) if(vus.Add(v)) { pile.Push(v); max=Math.Max(max,pile.Count); } }
    return (false,max);
}
// Le producteur BFS reconstruit le témoin à rebours à partir des parents.
List<int> Chemin(List<int>[] a,int source,int cible) {
    var file=new Queue<int>(); var parent=Enumerable.Repeat(-1,a.Length).ToArray(); file.Enqueue(source); parent[source]=source;
    while(file.Count>0) { int u=file.Dequeue(); if(u==cible) { var chemin=new List<int>();
        while(u!=source) { chemin.Add(u); u=parent[u]; } chemin.Reverse(); return chemin; }
        foreach(int v in a[u]) if(parent[v]<0) { parent[v]=u; file.Enqueue(v); } }
    return null;
}
// Le vérificateur lit les sommets une fois et rejette une arête absente ou un chemin trop long.
bool VerifierChemin(List<int>[] a,int source,int cible,IEnumerable<int> certificat) {
    int u=source, pas=0;
    foreach(int v in certificat) { if(++pas>a.Length || v<0 || v>=a.Length || !a[u].Contains(v)) return false; u=v; }
    return u==cible;
}
foreach(int n in new[]{40,100,250,600}) {
    var a=Graphe(n,4); var montre=System.Diagnostics.Stopwatch.StartNew(); var r=Dfs(a,0,n-1); montre.Stop();
    Console.WriteLine($"DFS n={n} atteint={r.Item1}, pile max={r.Item2}, temps={montre.Elapsed.TotalMilliseconds:F3} ms");
}
foreach(int n in new[]{40,100,250,600}) {
    var a=Graphe(n,2); var dfs=Dfs(a,0,n-1); var chemin=Chemin(a,0,n-1);
    bool verifie=chemin!=null && VerifierChemin(a,0,n-1,chemin);
    Console.WriteLine($"n={n} DFS={dfs.Item1}, certificat={verifie}, accord={dfs.Item1==verifie}, longueur={chemin?.Count ?? 0}");
    if(chemin!=null && chemin.Count>0) Console.WriteLine($"  témoin amputé refusé : {!VerifierChemin(a,0,n-1,chemin.Take(chemin.Count-1))}");
}

DFS n=40 atteint=True, pile max=8, temps=0.715 ms


DFS n=100 atteint=True, pile max=40, temps=0.492 ms


DFS n=250 atteint=True, pile max=109, temps=0.034 ms


DFS n=600 atteint=True, pile max=262, temps=0.109 ms


n=40 DFS=True, certificat=True, accord=True, longueur=2


  témoin amputé refusé : True


n=100 DFS=True, certificat=True, accord=True, longueur=3


  témoin amputé refusé : True


n=250 DFS=True, certificat=True, accord=True, longueur=7


  témoin amputé refusé : True


n=600 DFS=False, certificat=False, accord=True, longueur=0


L'accord est un contrôle sur ces graphes. Le témoin provient d'un BFS externe : son coût de construction n'est pas le coût de lecture du certificat.

## Exercice 1 — Deux chemins contradictoires dans 2-SAT

Dans le graphe d'implications, une formule est insatisfiable lorsqu'une variable et sa négation sont mutuellement atteignables. Vérifier un certificat formé de deux chemins opposés ; un seul chemin ne suffit pas à décider une formule générale.

In [3]:
bool? CertificatContradiction(List<int>[] implications,int x,int nonX,int[] aller,int[] retour) {
    // TODO : vérifier séparément les deux chemins orientés.
    return null;
}
var gEx=new[]{new List<int>{1},new List<int>{0,2},new List<int>()};
var ex1=CertificatContradiction(gEx,0,1,new[]{1},new[]{0});
if(ex1==null) Console.WriteLine("Exercice 1 à compléter");
else {
    if(ex1!=true || CertificatContradiction(gEx,0,1,new[]{1},new[]{2})!=false || CertificatContradiction(gEx,0,1,new[]{2},new[]{0})!=false)
        throw new Exception("Le certificat doit vérifier les deux chemins");
    Console.WriteLine("Deux chemins valides et deux témoins incorrects vérifiés");
}

Exercice 1 à compléter


## 2. 2-SAT et 3-SAT

Kosaraju calcule les composantes fortement connexes du graphe d'implications 2-SAT. DPLL simplifie par propagation unitaire et branche sur un littéral pour 3-SAT. Les familles tirées et leurs tailles suivent le source ; les chronos sont propres à l'environnement.

In [4]:
// Chaque clause (a ou b) fournit les implications non-a -> b et non-b -> a.
(bool,int[]) DeuxSat(int[][] clauses,int n) {
    int Id(int l)=>l>0?l-1:-l-1+n; int Neg(int i)=>i<n?i+n:i-n;
    var a=Enumerable.Range(0,2*n).Select(_=>new List<int>()).ToArray();
    var b=Enumerable.Range(0,2*n).Select(_=>new List<int>()).ToArray();
    foreach(var c in clauses) { int x=Id(c[0]),y=Id(c[1]); a[Neg(x)].Add(y); a[Neg(y)].Add(x); b[y].Add(Neg(x)); b[x].Add(Neg(y)); }
    // Kosaraju : ordre de fin sur le graphe, puis composantes sur le graphe transposé.
    var vu=new bool[2*n]; var ordre=new List<int>();
    void Visite(int u) { vu[u]=true; foreach(int v in a[u]) if(!vu[v]) Visite(v); ordre.Add(u); }
    for(int i=0;i<2*n;i++) if(!vu[i]) Visite(i);
    var comp=Enumerable.Repeat(-1,2*n).ToArray();
    void Attribue(int u,int c) { comp[u]=c; foreach(int v in b[u]) if(comp[v]<0) Attribue(v,c); }
    int id=0; ordre.Reverse(); foreach(int u in ordre) if(comp[u]<0) Attribue(u,id++);
    if(Enumerable.Range(0,n).Any(i=>comp[i]==comp[i+n])) return (false,null);
    return (true,Enumerable.Range(0,n).Select(i=>comp[i]>comp[i+n]?1:0).ToArray());
}
// DPLL compte chaque nœud ; une clause vide est une contradiction immédiate.
(bool,Dictionary<int,bool>,int) TroisSat(int[][] clauses) {
    int noeuds=0;
    Dictionary<int,bool> Resoudre(List<int[]> restantes,Dictionary<int,bool> partiel) {
        noeuds++;
        while(true) {
            if(restantes.Any(c=>c.Length==0)) return null;
            if(restantes.Count==0) return partiel;
            var unit=restantes.FirstOrDefault(c=>c.Length==1); if(unit==null) break;
            int l=unit[0]; if(partiel.TryGetValue(Math.Abs(l),out bool ancien) && ancien!=(l>0)) return null;
            partiel=new Dictionary<int,bool>(partiel); partiel[Math.Abs(l)]=l>0;
            restantes=restantes.Where(c=>!c.Contains(l)).Select(c=>c.Where(x=>x!=-l).ToArray()).ToList();
        }
        // La propagation est épuisée : explorer les deux valeurs du littéral choisi.
        int choix=restantes[0][0];
        foreach(int l in new[]{choix,-choix}) {
            var copie=new Dictionary<int,bool>(partiel) {[Math.Abs(l)]=l>0};
            var suite=restantes.Where(c=>!c.Contains(l)).Select(c=>c.Where(x=>x!=-l).ToArray()).ToList();
            var solution=Resoudre(suite,copie); if(solution!=null) return solution;
        }
        return null;
    }
    var solution=Resoudre(clauses.ToList(),new Dictionary<int,bool>()); return (solution!=null,solution,noeuds);
}
bool Verifier3(int[][] clauses,Dictionary<int,bool> solution) =>
    clauses.All(c=>c.Any(l=>solution.TryGetValue(Math.Abs(l),out bool v) && v==(l>0)));
int[][] Generer(int n,int m,int largeur) {
    var clauses=new int[m][];
    for(int i=0;i<m;i++) { var vars=largeur==2 ? new[]{rng.Next(1,n+1),rng.Next(1,n+1)} : Enumerable.Range(1,n).OrderBy(_=>rng.Next()).Take(largeur).ToArray(); clauses[i]=vars.Select(x=>rng.Next(2)==0?x:-x).ToArray(); }
    return clauses;
}
Console.WriteLine("Solveurs 2-SAT et DPLL définis ; une clause vide est explicitement insatisfiable");

Solveurs 2-SAT et DPLL définis ; une clause vide est explicitement insatisfiable


La propagation préserve une contradiction lorsqu'une clause devient vide ; le certificat 3-SAT est vérifié séparément sur la formule d'origine.

In [5]:
// Même protocole que la source : trois chronos par instance et moyenne des minima.
var montres2=new Dictionary<int,double>();
foreach(int n in new[]{50,200,800}) {
    double cumul=0;
    for(int essai=0;essai<5;essai++) { var f=Generer(n,3*n,2); double meilleur=double.PositiveInfinity;
        for(int rep=0;rep<3;rep++) { var sw=System.Diagnostics.Stopwatch.StartNew(); DeuxSat(f,n); sw.Stop(); meilleur=Math.Min(meilleur,sw.Elapsed.TotalMilliseconds); } cumul+=meilleur; }
    montres2[n]=cumul/5; Console.WriteLine($"2-SAT n={n}, moyenne des minima (5 instances × 3) : {cumul/5:F3} ms");
}
// La densité 4,6n et le nombre d’instances sont conservés pour chaque taille.
var noeudsMoyens=new Dictionary<int,double>(); var tempsMoyens=new Dictionary<int,double>();
foreach(int n in new[]{10,15,20,25,30}) {
    double temps=0,noeuds=0;
    for(int essai=0;essai<20;essai++) { var f=Generer(n,(int)(4.6*n),3); double meilleur=double.PositiveInfinity; int count=0;
        for(int rep=0;rep<3;rep++) { var sw=System.Diagnostics.Stopwatch.StartNew(); var sol=TroisSat(f); sw.Stop(); meilleur=Math.Min(meilleur,sw.Elapsed.TotalMilliseconds); count=sol.Item3; }
        temps+=meilleur; noeuds+=count; }
    noeudsMoyens[n]=noeuds/20; tempsMoyens[n]=temps/20;
    Console.WriteLine($"3-SAT n={n}, 20 instances × 3 : noeuds moyens={noeuds/20:F1}, temps moyen={temps/20:F3} ms");
}
// La solution fournie par DPLL est contrôlée sur la formule initiale.
int certificats=0;
for(int essai=0;essai<20;essai++) { int[][] f=null; (bool,Dictionary<int,bool>,int) sol=default;
    for(int tentative=0;tentative<12;tentative++) { f=Generer(12,48,3); sol=TroisSat(f); if(sol.Item1) break; }
    if(sol.Item1 && Verifier3(f,sol.Item2)) certificats++;
}
double facteurNoeuds=Math.Pow(noeudsMoyens[30]/noeudsMoyens[10],1.0/20);
double facteurTemps=Math.Pow(tempsMoyens[30]/tempsMoyens[10],1.0/20);
Console.WriteLine($"Certificats vérifiés {certificats}/20 ; facteur estimé par variable : noeuds={facteurNoeuds:F3}, temps={facteurTemps:F3}");

2-SAT n=50, moyenne des minima (5 instances × 3) : 0.030 ms


2-SAT n=200, moyenne des minima (5 instances × 3) : 0.141 ms


2-SAT n=800, moyenne des minima (5 instances × 3) : 0.583 ms


3-SAT n=10, 20 instances × 3 : noeuds moyens=9.2, temps moyen=0.246 ms


3-SAT n=15, 20 instances × 3 : noeuds moyens=14.4, temps moyen=0.695 ms


3-SAT n=20, 20 instances × 3 : noeuds moyens=25.8, temps moyen=3.966 ms


3-SAT n=25, 20 instances × 3 : noeuds moyens=37.2, temps moyen=12.434 ms


3-SAT n=30, 20 instances × 3 : noeuds moyens=57.8, temps moyen=27.490 ms


Certificats vérifiés 20/20 ; facteur estimé par variable : noeuds=1.097, temps=1.266


Les nombres de nœuds dépendent des instances et de la stratégie ; les temps dépendent aussi de la machine. Ces points finis illustrent les algorithmes et ne démontrent pas leurs complexités asymptotiques.

## Exercice 2 — Projeter les temps DPLL

À partir du dernier temps mesuré et du facteur par variable, projeter n=40 et n=60 puis calculer le premier n dépassant une heure. Si le facteur vaut au plus 1, indiquer qu'aucun dépassement ne découle de cette extrapolation.

In [6]:
(double,double,int?)? Projection(double temps30Secondes,double facteur) {
    // TODO : projeter et ne renvoyer une taille finie que si le facteur le permet.
    return null;
}
var ex2=Projection(1.0,2.0);
if(ex2==null) Console.WriteLine("Exercice 2 à compléter");
else {
    var p=ex2.Value;
    if(Math.Abs(p.Item1-1024)>1e-8 || Math.Abs(p.Item2-Math.Pow(2,30))>1e-3 || p.Item3!=42 || Projection(1,1).Value.Item3!=null)
        throw new Exception("Projection incorrecte");
    Console.WriteLine("Projection et absence de dépassement si facteur 1 vérifiées");
    var reel=Projection(tempsMoyens[30]/1000.0,facteurTemps);
    if(reel==null) throw new Exception("Projection absente sur les mesures");
    double baseReelle=tempsMoyens[30]/1000.0;
    if(Math.Abs(reel.Value.Item1-baseReelle*Math.Pow(facteurTemps,10))>1e-8 || Math.Abs(reel.Value.Item2-baseReelle*Math.Pow(facteurTemps,30))>1e-8)
        throw new Exception("Projection des mesures incorrecte");
    int? seuilAttendu=null;
    if(baseReelle>3600) seuilAttendu=30;
    else if(facteurTemps>1) { int n=30; double temps=baseReelle; while(temps<=3600 && n<10000) { temps*=facteurTemps; n++; } if(temps>3600) seuilAttendu=n; }
    if(reel.Value.Item3!=seuilAttendu) throw new Exception("Seuil d'une heure incorrect sur les mesures");
    Console.WriteLine($"Mesures réelles : n=40 {reel.Value.Item1:F4}s, n=60 {reel.Value.Item2:F4}s, seuil 1h n={reel.Value.Item3?.ToString() ?? "non atteint"}");
}

Exercice 2 à compléter


## 3. Deux jeux d'oracles finis

Dans le premier jeu, chaque machine inspecte au plus n témoins. On choisit un témoin non interrogé pour démentir une réponse négative, ou aucun témoin pour démentir une réponse positive. Dans le second, une requête délègue une recherche existentielle à l'oracle. Ces exemples finis illustrent la relativisation ; ils ne construisent pas les oracles infinis du théorème général.

In [7]:
int MachineP(HashSet<string> oracle,string x,int defaut) {
    int n=x.Length, positives=0;
    for(int y=0;y<n;y++) if(oracle.Contains(x+"#"+Convert.ToString(y,2).PadLeft(n,'0'))) positives++;
    return positives==0?defaut:(positives>n/2?1:0);
}
int Langage(HashSet<string> oracle,string x) {
    int n=x.Length; for(int y=0;y<(1<<n);y++) if(oracle.Contains(x+"#"+Convert.ToString(y,2).PadLeft(n,'0'))) return 1; return 0;
}
var oracle=new HashSet<string>(); var stades=new List<(int i,int n,int reponse,int reel)>();
for(int i=1;i<=4;i++) {
    int n=6+2*i; string x=Convert.ToString(i,2).PadLeft(n,'0'); int rep=MachineP(oracle,x,i%2);
    if(rep==0) { int y=(i*7919)%(1<<n); if(y<n) y=n; oracle.Add(x+"#"+Convert.ToString(y,2).PadLeft(n,'0')); }
    int reel=Langage(oracle,x); stades.Add((i,n,rep,reel));
    Console.WriteLine($"M{i}, n={n}, réponse={rep}, témoin exhaustif={reel}, contredite={rep!=reel}");
}
bool Predicat(string x,string y)=>x.Count(c=>c=='1')+y.Count(c=>c=='1')>=x.Length;
int OracleExistentiel(string x) { for(int y=0;y<(1<<x.Length);y++) if(Predicat(x,Convert.ToString(y,2).PadLeft(x.Length,'0'))) return 1; return 0; }
foreach(int i in new[]{0,5,13,21,31}) { string x=Convert.ToString(i,2).PadLeft(5,'0');
    int reponse=OracleExistentiel(x); int exhaustif=Enumerable.Range(0,32).Any(y=>Predicat(x,Convert.ToString(y,2).PadLeft(5,'0')))?1:0;
    Console.WriteLine($"x={x}, oracle une requête={reponse}, recherche exhaustive={exhaustif}"); }
Console.WriteLine("Quatre stades finis et cinq entrées : illustrations seulement");
// Un second prédicat fournit une réponse positive et une négative.
bool PredicatMixte(string x,string y)=>x.All(c=>c=='0') && y.All(c=>c=='0');
foreach(string x in new[]{"00000","00101"}) {
    string temoin=Enumerable.Range(0,32).Select(y=>Convert.ToString(y,2).PadLeft(5,'0')).FirstOrDefault(y=>PredicatMixte(x,y));
    bool existe=temoin!=null;
    if(existe && !PredicatMixte(x,temoin)) throw new Exception("Témoin incohérent");
    Console.WriteLine($"Prédicat mixte x={x} : existe={existe}, témoin={temoin ?? "aucun"}");
}

M1, n=8, réponse=1, témoin exhaustif=0, contredite=True


M2, n=10, réponse=0, témoin exhaustif=1, contredite=True


M3, n=12, réponse=1, témoin exhaustif=0, contredite=True


M4, n=14, réponse=0, témoin exhaustif=1, contredite=True


x=00000, oracle une requête=1, recherche exhaustive=1


x=00101, oracle une requête=1, recherche exhaustive=1


... (2 outputs left out)

x=11111, oracle une requête=1, recherche exhaustive=1


Quatre stades finis et cinq entrées : illustrations seulement


Prédicat mixte x=00000 : existe=True, témoin=00000


Prédicat mixte x=00101 : existe=False, témoin=aucun


Le premier jeu confirme les désaccords pour quatre machines précises ; le second ne compare qu'un prédicat sur cinq entrées. Aucun ne prouve une égalité ou une séparation de classes entières.

## Exercice 3 — Ajouter un stade

Étendre le premier jeu pour une cinquième machine à une longueur fraîche. Choisir un témoin hors des n premières requêtes si nécessaire et vérifier que les quatre résultats précédents restent inchangés.

In [8]:
(HashSet<string>,List<(int,int,int,int)>)? EtendreOracle(HashSet<string> courant,int indice) {
    // TODO : copier l'oracle, choisir la longueur fraîche et vérifier le nouveau stade.
    return null;
}
var ex3=EtendreOracle(oracle,5);
if(ex3==null) Console.WriteLine("Exercice 3 à compléter");
else {
    var (nouveau,historique)=ex3.Value; int n=16; string x=Convert.ToString(5,2).PadLeft(n,'0');
    if(nouveau==null || historique==null || !historique.Any(s=>s.Item1==5 && s.Item2==16) || MachineP(nouveau,x,1)==Langage(nouveau,x) ||
       stades.Any(s=>Langage(nouveau,Convert.ToString(s.i,2).PadLeft(s.n,'0'))!=s.reel))
        throw new Exception("Le stade ne contredit pas la machine ou modifie les anciens");
    Console.WriteLine("Cinquième stade et anciens résultats vérifiés");
}

Exercice 3 à compléter


## 4. Carte du Zoo et limites

La carte distingue des ressources et des modes de calcul. **L** regroupe les problèmes décidables en espace logarithmique déterministe ; **NL** autorise le nondéterminisme avec le même ordre d'espace ; **P** regroupe les problèmes décidables en temps polynomial déterministe ; **NP** ceux dont un certificat polynomial se vérifie en temps polynomial ; **coNP** contient les complémentaires des problèmes de NP. **PSPACE** autorise un espace polynomial, et **EXPTIME** un temps exponentiel.

| Relation | Statut rapporté par le source Python | Illustration ici |
|---|---|---|
| L ⊆ NL | Inclusion établie | certificat de chemin lu une fois ; le BFS qui le construit est hors du vérificateur |
| NL ⊆ P | Inclusion établie | atteignabilité décidée par parcours déterministe |
| P ⊆ NP | Inclusion établie | vérification de certificat 3-SAT |
| NP ⊆ PSPACE | Inclusion établie | citée, non démontrée par ce carnet |
| P ⊊ EXPTIME | Hiérarchie en temps | citée, non démontrée par ce carnet |
| P vs NP | Ouvert dans le source | jeux d'oracles finis, pas une preuve générale |
| NP vs coNP | Ouvert dans le source | la non-satisfiabilité ne dispose pas d'un certificat polynomial général connu |
| NP vs PSPACE | Ouvert dans le source | l'égalité IP = PSPACE n'établit pas cette séparation |

La barrière de relativisation est rapportée par le source : l'existence de deux oracles infinis de comportements opposés limite les preuves qui se relativisent. Les jeux finis de §3 n'établissent ni ces oracles infinis ni une séparation de classes. L'encart Mathlib du source (constat historique) distingue la calculabilité formalisée des classes de complexité bornées, sans en déduire une absence actuelle exhaustive.

## À retenir

- Lire un certificat et le construire sont deux tâches différentes.
- DPLL et Kosaraju sont exécutés sur des familles et budgets explicités.
- Les jeux d'oracles finis illustrent une idée sans établir de séparation de classes.

## Vérifiez votre compréhension

1. Pourquoi un certificat amputé doit-il être rejeté ?
2. Que mesure le compteur de nœuds DPLL ?
3. Pourquoi quatre machines battues ne prouvent-elles pas une séparation infinie ?

<details><summary>Éléments de réponse</summary>

1. Il ne rejoint plus la cible. 2. Les appels explorés sur une instance par une stratégie donnée. 3. Le théorème concerne toutes les machines et des tailles non bornées.
</details>

## Pour aller plus loin

[Source Python](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) et [03b](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb). Les références aux résultats historiques reprennent le carnet Python, sans revendiquer une lecture directe des publications.